In [ ]:
#
# Universidad EAFIT
# 2026-2
# SI7016 - NLP - Lecture 05b - Ejercicio: Chatbot RAG (actualizado 2026-2)
#

In [1]:
# instalar dependencias
%pip install langchain-openai langchain-chroma langgraph streamlit chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/6

In [2]:
# paso 1: modelo de chat
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

# La API key se toma de una variable de entorno, nunca hardcodeada en el notebook
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Modelo de frontera 2026 (antes: "gpt-4")
llm = ChatOpenAI(model="gpt-5.6")

OpenAI API Key: ··········


In [3]:
# Inicializar el Vector Store (Chroma) con embeddings de OpenAI
#
# Nota 2026: en el notebook original se creaba un chromadb.PersistentClient
# "crudo" por un lado y, más adelante, un Chroma de LangChain por otro,
# apuntando al mismo directorio pero NO a la misma colección - los documentos
# agregados con el cliente crudo (con el embedding por defecto de chromadb)
# quedaban invisibles para el retriever de LangChain (que buscaba con
# embeddings de OpenAI en otra colección). Aquí se usa un único cliente
# (langchain_chroma.Chroma) tanto para escribir como para leer.
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(
    collection_name="chatbot_knowledge",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

In [4]:
# Agregar Datos de Referencia a la Base de Datos

documents = [
    "La inteligencia artificial es el campo de la informática que estudia cómo crear sistemas capaces de realizar tareas que requieren inteligencia humana.",
    "GPT-5.6 es un modelo de lenguaje de OpenAI basado en la arquitectura de transformers.",
    "LangChain es una biblioteca para desarrollar aplicaciones basadas en modelos de lenguaje.",
]

vectorstore.add_texts(documents, ids=["1", "2", "3"])

['1', '2', '3']

In [5]:
# Crear el retriever para buscar documentos relevantes
retriever = vectorstore.as_retriever()

# Prueba rápida de recuperación
retriever.invoke("¿Qué es LangChain?")

[Document(id='3', metadata={}, page_content='LangChain es una biblioteca para desarrollar aplicaciones basadas en modelos de lenguaje.'),
 Document(id='2', metadata={}, page_content='GPT-5.6 es un modelo de lenguaje de OpenAI basado en la arquitectura de transformers.'),
 Document(id='1', metadata={}, page_content='La inteligencia artificial es el campo de la informática que estudia cómo crear sistemas capaces de realizar tareas que requieren inteligencia humana.')]

## Paso 2: memoria de la conversación

`ConversationBufferMemory` + `ConversationalRetrievalChain` están **deprecados** en LangChain (siguen funcionando con warnings, pero no reciben mejoras y LangChain recomienda migrar). El reemplazo moderno para un chatbot con memoria es un grafo de **LangGraph**: un nodo recupera contexto y genera la respuesta, y un `checkpointer` guarda el historial por `thread_id` (mismo patrón usado en los labs de agentes de este curso, ej. `class03a-langchain-agent.ipynb`).

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver


def retrieve_and_generate(state: MessagesState):
    user_message = state["messages"][-1].content
    docs = retriever.invoke(user_message)
    context = "\n\n".join(doc.page_content for doc in docs) or "(sin contexto relevante)"
    system = SystemMessage(
        content=f"Responde la pregunta del usuario usando este contexto cuando sea relevante:\n\n{context}"
    )
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


graph_builder = StateGraph(MessagesState)
graph_builder.add_node("retrieve_and_generate", retrieve_and_generate)
graph_builder.add_edge(START, "retrieve_and_generate")
graph_builder.add_edge("retrieve_and_generate", END)

# checkpointer en memoria: guarda el historial de cada conversación (thread_id)
chat_chain = graph_builder.compile(checkpointer=InMemorySaver())

## Paso 3: función del chatbot

`thread_id` identifica una conversación - todos los mensajes con el mismo `thread_id` comparten memoria; uno distinto empieza una conversación nueva.

In [7]:
def chat_with_gpt(user_input, thread_id="demo"):
    config = {"configurable": {"thread_id": thread_id}}
    result = chat_chain.invoke({"messages": [HumanMessage(content=user_input)]}, config=config)
    return result["messages"][-1].content


# Prueba en el notebook (sin Streamlit): dos turnos en el mismo hilo
print(chat_with_gpt("¿Qué es LangChain?"))
print(chat_with_gpt("¿Y para qué sirve lo que acabas de explicarme?"))  # usa la memoria del turno anterior

LangChain es una biblioteca para desarrollar aplicaciones basadas en modelos de lenguaje. Facilita conectar estos modelos con fuentes de datos, herramientas externas, memoria y flujos de trabajo para crear chatbots, agentes, sistemas de preguntas y respuestas y otras soluciones de IA.
LangChain sirve para **crear aplicaciones prácticas con modelos de lenguaje**, conectándolos con datos y herramientas externas.

Por ejemplo, permite desarrollar:

- **Chatbots** que recuerdan el contexto de una conversación.
- **Asistentes** que consultan documentos, bases de datos o páginas web.
- **Sistemas de preguntas y respuestas** sobre información propia de una empresa.
- **Agentes** capaces de usar herramientas, llamar a APIs o ejecutar tareas.
- **Flujos automatizados**, como resumir textos, clasificar mensajes o generar informes.

En resumen, ayuda a pasar de usar un modelo de lenguaje de forma aislada a integrarlo en una aplicación completa.


In [ ]:
# Ejecución del Chatbot (interfaz Streamlit, en app.py)
!streamlit run app.py

## Mejoras propuestas

- Cargar documentos propios (PDF, páginas web) en vez de la lista de ejemplo - ver `class05-3ejercicio-ProyectoRAG-news.ipynb` para un caso con un dataset real.
- Evaluar el sistema con RAGAS (Faithfulness, Answer Relevancy, Context Precision/Recall) en vez de solo probarlo a mano.
- Integrar APIs externas (ej. Wikipedia) como herramientas adicionales del agente - ver `class05c.ipynb` (Agentes) para el patrón de tool calling.
- Agregar entrada de voz (Speech-to-Text) para un chatbot conversacional por voz.